In [201]:
import pandas as pd
import torch

In [202]:
df = pd.read_csv("IMT2024007_train_var1.csv")
tdf = pd.read_csv("IMT2024007_test_var1.csv")
tdf

,x1,x2,x3,x4,x5,x6
0,1.000000,0.811058,0.222203,1.000000,1.000000,-0.565605
1,-0.450541,-1.000000,-0.198250,-0.334777,1.000000,-1.000000
2,-1.000000,0.319095,-1.000000,1.000000,-1.000000,-0.670666
3,1.000000,1.000000,1.000000,1.000000,0.731143,-1.000000
4,1.000000,0.322927,1.000000,0.061343,0.048063,-1.000000
...,...,...,...,...,...,...
995,0.885909,1.000000,0.295332,1.000000,-1.000000,-1.000000
996,-0.556144,0.270201,-1.000000,-1.000000,0.478120,0.410096
997,-1.000000,-0.717064,-0.763326,-0.687992,1.000000,0.771407
998,1.000000,1.000000,1.000000,0.809048,1.000000,1.000000


In [203]:
x = df.drop("y",axis = 1).values
y = df["y"].values

x = torch.tensor(x,dtype = torch.float32)
y = torch.tensor(y,dtype = torch.float32)

In [204]:
num_samples = x.shape[0]                # gives number of rows
indices = torch.randperm(num_samples)   # generates a random permutation of the "row numbers"

train_size = int(0.9*num_samples)      
train_idx = indices[:train_size]         #90% of data in the shuffled dataset is used for training 
val_idx = indices[train_size:]          #rest of the data for validation!

x_train = x[train_idx]
y_train = y[train_idx]

x_val = x[val_idx]
y_val = y[val_idx]


In [205]:
def polynomial(x,n):
    samples,features = x.shape
    bias_col = torch.ones(samples, dtype=x.dtype)
    poly_terms = [bias_col]
    current_level = []
    #Degree 1
    for i in range (features):
        col = x[ :, i]
        poly_terms.append(col)
        current_level.append((col,i))

    for degree in range (2,n+1):
        new_level = []

        for term,indx in current_level:
            for j in range (indx,features):
                new_term = term * x[ : , j]

                poly_terms.append(new_term)
                new_level.append((new_term,j))
        current_level = new_level
    return torch.stack(poly_terms, dim=1)

In [206]:
# Lists to store the error values for plotting later
train_mse_list = []
val_mse_list = []
val_r2_list = []
degrees = list(range(1, 11))

y_train_mean = torch.mean(y_train)
train_total_variance = torch.sum((y_train - y_train_mean) ** 2)

y_val_mean = torch.mean(y_val)
val_total_variance = torch.sum((y_val - y_val_mean) ** 2)

for d in degrees:
    # Expanding the features for both datasets
    A_train = polynomial(x_train, d)
    A_val = polynomial(x_val, d)
    
    # Calculating best weights using pseudo-inverse as discussed in class
    w = torch.pinverse(A_train) @ y_train
    
    # predictions
    y_train_pred = A_train @ w
    y_val_pred = A_val @ w
    
    # calculations of metrics(MSE,R^2)
    train_mse = torch.mean((y_train - y_train_pred)**2).item()
    val_mse = torch.mean((y_val - y_val_pred)**2).item()

    
    train_model_error = torch.sum((y_train - y_train_pred) ** 2)
    train_r2 = 1 - (train_model_error / train_total_variance).item()
    
    val_model_error = torch.sum((y_val - y_val_pred) ** 2)
    val_r2 = 1 - (val_model_error / val_total_variance).item()
    
    # 5. Save the results
    train_mse_list.append(train_mse)
    val_mse_list.append(val_mse)
    val_r2_list.append(val_r2)
    # Print a clean summary for each degree
    print(f"Degree {d:2d} | Train MSE: {train_mse:10.4f} (R2: {train_r2:7.4f}) | Val MSE: {val_mse:10.4f} (R2: {val_r2:7.4f})")

Degree  1 | Train MSE:     8.9146 (R2:  0.1206) | Val MSE:     9.0819 (R2:  0.0905)
Degree  2 | Train MSE:     2.6289 (R2:  0.7407) | Val MSE:     3.1589 (R2:  0.6837)
Degree  3 | Train MSE:     0.7837 (R2:  0.9227) | Val MSE:     0.9159 (R2:  0.9083)
Degree  4 | Train MSE:     0.3387 (R2:  0.9666) | Val MSE:     0.6571 (R2:  0.9342)
Degree  5 | Train MSE:     0.1255 (R2:  0.9876) | Val MSE:     0.7191 (R2:  0.9280)
Degree  6 | Train MSE:     0.0000 (R2:  1.0000) | Val MSE:   404.7671 (R2: -39.5354)
Degree  7 | Train MSE:     0.0000 (R2:  1.0000) | Val MSE:     5.6039 (R2:  0.4388)
Degree  8 | Train MSE:     0.0000 (R2:  1.0000) | Val MSE:     3.0968 (R2:  0.6899)
Degree  9 | Train MSE:     0.0000 (R2:  1.0000) | Val MSE:     3.2217 (R2:  0.6774)
Degree 10 | Train MSE:     0.0000 (R2:  1.0000) | Val MSE:     2.5853 (R2:  0.7411)


In [207]:
"""
   Looking at these results, it's clearly evident that the model stops learning after a certain
   degree(5) and starts memorizing after that, the proof is the val MSE and the R^2 , so i'll be choosing 
   degree 4 polynomial as my final model as it avoids overfitting and have the best val MSE and do the appropriate testing
"""



"\n   Looking at these results, it's clearly evident that the model stops learning after a certain\n   degree(5) and starts memorizing after that, the proof is the val MSE and the R^2 , so i'll be choosing \n   degree 4 polynomial as my final model as it avoids overfitting and have the best val MSE and do the appropriate testing\n"

In [208]:
A_train = polynomial(x,4)
w = torch.pinverse(A_train) @ y

In [209]:
x_test = tdf.values
x_test = torch.tensor(x_test,dtype = torch.float32)
A_test = polynomial(x_test,4)
y_test = A_test @ w

In [210]:
pred_df = pd.DataFrame(y_test.numpy(),columns = ["y"])
pred_df.to_csv("IMT2024007_pred_var_1.csv",index = False)